# Step 6 only — benchmark the already-trained QAT checkpoints

Standalone notebook. Does **not** re-run pruning, recovery, PTQ, or QAT training --
those already completed successfully and their checkpoints are already saved on
Drive (`ma1dcnn_trial13_prunedXX_recovered.pt` and `..._quantized_qat.pt` for
ratios 20%/40%/60%), confirmed present by a direct Drive file-existence check.

This notebook just: mounts Drive, defines the handful of classes needed to load
those checkpoints back in (`MA1DCNN`'s pieces, plus `QuantizedConvTrunk` -- Python's
pickle needs these class definitions available to reconstruct the saved objects, the
same reason every real-application cell in the main notebook needs them too),
re-derives each checkpoint's best operating threshold, and benchmarks macro-F1,
parameter count, serialized size, and CPU latency -- the real Step 6 numbers.

Everything here is copied verbatim from the already-verified full notebook
(`04_compression_pruning_quant.ipynb`, Sections 0, 1, 10.1, 10.3, 11.1, 11.2) --
nothing new is being written or re-derived, just re-assembled into a lighter
notebook that doesn't need the earlier sections to have run first.

## 0. Imports

In [1]:
from __future__ import annotations

import torch
import torch.nn as nn

CAM_REDUCTION_RATIO = 4


## 1. Model — repeated verbatim from `ma1dcnn.ipynb` / `ma1dcnn.py`

Needed because `QuantizedConvTrunk` (below) holds `ChannelAttentionModule` /
`ElementwiseAttentionModule` instances by reference -- unpickling the saved
checkpoint needs these class definitions in scope, the same way every other
real-application cell in the main notebook needs them.

In [2]:
class Conv1dSamePadding(nn.Module):
    """1D convolution that keeps the sequence length unchanged (stride=1, odd kernel)."""

    def __init__(self, in_channels, out_channels, kernel_size, conv_dropout=0.0, use_batchnorm=False):
        super().__init__()
        if kernel_size % 2 == 0:
            raise ValueError(f"Conv1dSamePadding needs an odd kernel_size, got {kernel_size}")
        padding = (kernel_size - 1) // 2  # same-padding formula: P = (K - 1) / 2
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, stride=1, padding=padding)
        self.bn = nn.BatchNorm1d(out_channels) if use_batchnorm else nn.Identity()
        self.activation = nn.ReLU()
        self.conv_dropout = nn.Dropout(p=conv_dropout)

    def forward(self, x):
        return self.conv_dropout(self.activation(self.bn(self.conv(x))))


class ChannelAttentionModule(nn.Module):
    """CAM: one learned attention weight per CHANNEL, shared across all timesteps."""

    def __init__(self, n_channels, reduction_ratio=CAM_REDUCTION_RATIO):
        super().__init__()
        hidden = max(1, n_channels // reduction_ratio)
        self.squeeze = nn.AdaptiveAvgPool1d(1)
        self.excite = nn.Sequential(
            nn.Linear(n_channels, hidden), nn.ReLU(),
            nn.Linear(hidden, n_channels), nn.Sigmoid(),
        )

    def forward(self, x):
        batch, channels, length = x.shape
        squeezed = self.squeeze(x).view(batch, channels)
        channel_weights = self.excite(squeezed).view(batch, channels, 1)
        return x * channel_weights


class ElementwiseAttentionModule(nn.Module):
    """EAM: one learned attention weight per TIMESTEP, shared across all channels."""

    def __init__(self, n_channels):
        super().__init__()
        self.score = nn.Conv1d(in_channels=n_channels, out_channels=1, kernel_size=1)
        self.activation = nn.Sigmoid()

    def forward(self, x):
        attention = self.activation(self.score(x))
        return x * attention


class MA1DCNN(nn.Module):
    def __init__(self, n_input_channels, channel_1, channel_2, channel_3,
                 kernel_size, fc_units, dropout, n_outputs,
                 cam_reduction_ratio=CAM_REDUCTION_RATIO, conv_dropout=0.0, use_batchnorm=False):
        super().__init__()
        self.conv1 = Conv1dSamePadding(n_input_channels, channel_1, kernel_size, conv_dropout, use_batchnorm)
        self.conv2 = Conv1dSamePadding(channel_1, channel_2, kernel_size, conv_dropout, use_batchnorm)
        self.conv3 = Conv1dSamePadding(channel_2, channel_3, kernel_size, conv_dropout, use_batchnorm)
        self.cam = ChannelAttentionModule(channel_3, reduction_ratio=cam_reduction_ratio)
        self.eam = ElementwiseAttentionModule(channel_3)
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(p=dropout)
        self.fc = nn.Linear(channel_3, fc_units)
        self.fc_activation = nn.ReLU()
        self.output_layer = nn.Linear(fc_units, n_outputs)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.cam(x)
        x = self.eam(x)
        x = self.global_avg_pool(x)
        x = x.view(x.shape[0], -1)
        x = self.dropout(x)
        x = self.fc(x)
        x = self.fc_activation(x)
        return self.output_layer(x)

    def predict_proba(self, x):
        with torch.no_grad():
            return torch.sigmoid(self.forward(x))


def build_ma1dcnn(config):
    return MA1DCNN(
        n_input_channels=config["n_input_channels"], channel_1=config["channel_1"],
        channel_2=config["channel_2"], channel_3=config["channel_3"],
        kernel_size=config["kernel_size"], fc_units=config["fc_units"],
        dropout=config["dropout"], n_outputs=config["n_outputs"],
        conv_dropout=config.get("conv_dropout", 0.0),
        use_batchnorm=config.get("use_batchnorm", False),
    )


## 2. `QuantizedConvTrunk` — repeated verbatim from Section 10.1

The class the saved QAT checkpoint actually is an instance of. Needed in scope for
`torch.load` to reconstruct it -- this is exactly what unpickling a whole `nn.Module`
object (rather than a plain `state_dict`) requires.

In [3]:
class QuantizedConvTrunk(nn.Module):
    """Wraps an already-recovered MA1DCNN for static PTQ/QAT of conv1/conv2/conv3 only."""

    def __init__(self, model):
        super().__init__()
        self.quant = torch.quantization.QuantStub()
        self.conv1 = model.conv1
        self.conv2 = model.conv2
        self.conv3 = model.conv3
        self.dequant = torch.quantization.DeQuantStub()
        self.cam = model.cam
        self.eam = model.eam
        self.global_avg_pool = model.global_avg_pool
        self.dropout = model.dropout
        self.fc = model.fc
        self.fc_activation = model.fc_activation
        self.output_layer = model.output_layer

    def forward(self, x):
        x = self.quant(x)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.dequant(x)
        x = self.cam(x)
        x = self.eam(x)
        x = self.global_avg_pool(x)
        x = x.view(x.shape[0], -1)
        x = self.dropout(x)
        x = self.fc(x)
        x = self.fc_activation(x)
        return self.output_layer(x)


## 3. Mount Drive + get `train.py` / `models/ma1dcnn.py` onto disk

Same mechanism every notebook in this project uses: a fresh Colab session's
`/content/` doesn't have these files yet, so they're written out via `%%writefile`
(embedding the real, unmodified source) in the exact layout `train.py`'s own
`from models.ma1dcnn import ...` expects.

In [4]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data"
REAL_WINDOWED_DIR = f"{DATA_DIR}/windowed"

import os
os.makedirs("/content/edge_fault_dx/models", exist_ok=True)
os.makedirs("/content/edge_fault_dx/training", exist_ok=True)


Mounted at /content/drive


In [5]:
%%writefile /content/edge_fault_dx/models/ma1dcnn.py
"""MA1DCNN architecture -- src/edge_fault_dx/models/ma1dcnn.py.

Moved here from training/train.py, where it originally lived inline per this
project's "every script runs standalone, nothing imported from elsewhere in
the codebase" discipline (so each script could be self-check-verified in
isolation). That discipline served the proving-out phase; now that trial13 is
confirmed (0.8401 macro-F1, four independent runs -- see train.py's own
module docstring), the architecture belongs in its own importable module,
matching this repo's scaffold (src/edge_fault_dx/models/, separate from
training/, compression/, evaluation/, deployment/).

Nothing about the architecture itself changed in this move -- every class and
function below is byte-for-byte identical to what train.py defined inline.
train.py now imports build_ma1dcnn (and MA1DCNN) from here instead of
defining them, and re-exports both so `from train import build_ma1dcnn` --
which threshold_sweep.py and false_alarm_review.py both rely on -- keeps
working unchanged.

Per this repo's own scaffold comment (see models/ma1dcnn.py's prior TODO
stub): architecture per Wang et al. (2020) -- full bibliographic details
(journal/venue, volume, pages) still unverified; see the praxis write-up's
References section for that open item.
"""
from __future__ import annotations

import torch
import torch.nn as nn

CAM_REDUCTION_RATIO = 4  # repeated from ma1dcnn.ipynb -- needed to construct MA1DCNN below


# =============================================================================
# Model -- repeated verbatim from ma1dcnn.ipynb / training/train.py
# =============================================================================

class Conv1dSamePadding(nn.Module):
    """1D convolution that keeps the sequence length unchanged (stride=1, odd kernel)."""

    def __init__(self, in_channels, out_channels, kernel_size, conv_dropout=0.0, use_batchnorm=False):
        super().__init__()
        if kernel_size % 2 == 0:
            raise ValueError(f"Conv1dSamePadding needs an odd kernel_size, got {kernel_size}")
        padding = (kernel_size - 1) // 2  # same-padding formula: P = (K - 1) / 2
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, stride=1, padding=padding)
        # Identity (zero params/buffers) when disabled (the default); placed BEFORE
        # activation -- normalizes the raw conv output, not an already-ReLU-clipped one.
        self.bn = nn.BatchNorm1d(out_channels) if use_batchnorm else nn.Identity()
        self.activation = nn.ReLU()
        self.conv_dropout = nn.Dropout(p=conv_dropout)  # p=0.0 default is a no-op, existing configs unaffected

    def forward(self, x):
        return self.conv_dropout(self.activation(self.bn(self.conv(x))))  # self.bn is a no-op when use_batchnorm=False


class ChannelAttentionModule(nn.Module):
    """CAM: one learned attention weight per CHANNEL, shared across all timesteps."""

    def __init__(self, n_channels, reduction_ratio=CAM_REDUCTION_RATIO):
        super().__init__()
        hidden = max(1, n_channels // reduction_ratio)
        self.squeeze = nn.AdaptiveAvgPool1d(1)
        self.excite = nn.Sequential(
            nn.Linear(n_channels, hidden), nn.ReLU(),
            nn.Linear(hidden, n_channels), nn.Sigmoid(),
        )

    def forward(self, x):
        batch, channels, length = x.shape
        squeezed = self.squeeze(x).view(batch, channels)
        channel_weights = self.excite(squeezed).view(batch, channels, 1)
        return x * channel_weights


class ElementwiseAttentionModule(nn.Module):
    """EAM: one learned attention weight per TIMESTEP, shared across all channels."""

    def __init__(self, n_channels):
        super().__init__()
        self.score = nn.Conv1d(in_channels=n_channels, out_channels=1, kernel_size=1)
        self.activation = nn.Sigmoid()

    def forward(self, x):
        attention = self.activation(self.score(x))
        return x * attention


class MA1DCNN(nn.Module):
    """3 conv blocks -> CAM -> EAM -> global average pooling -> FC head -> 5 logits."""

    def __init__(self, n_input_channels, channel_1, channel_2, channel_3,
                 kernel_size, fc_units, dropout, n_outputs,
                 cam_reduction_ratio=CAM_REDUCTION_RATIO, conv_dropout=0.0, use_batchnorm=False):
        super().__init__()
        # conv_dropout/use_batchnorm default to off -- BEFORE_TUNING_CONFIG/AFTER_TUNING_CONFIG/
        # BEFORE_TUNING_CONFIG_CONV_DROPOUT build unaffected models. The two are independent,
        # composable levers -- use_batchnorm=True layers ON TOP of conv_dropout, never replaces it.
        self.conv1 = Conv1dSamePadding(n_input_channels, channel_1, kernel_size, conv_dropout, use_batchnorm)
        self.conv2 = Conv1dSamePadding(channel_1, channel_2, kernel_size, conv_dropout, use_batchnorm)
        self.conv3 = Conv1dSamePadding(channel_2, channel_3, kernel_size, conv_dropout, use_batchnorm)
        self.cam = ChannelAttentionModule(channel_3, reduction_ratio=cam_reduction_ratio)
        self.eam = ElementwiseAttentionModule(channel_3)
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(p=dropout)
        self.fc = nn.Linear(channel_3, fc_units)
        self.fc_activation = nn.ReLU()
        self.output_layer = nn.Linear(fc_units, n_outputs)

    def forward(self, x):
        # x: (batch, n_input_channels, L) -- a batch of windowed, normalized sensor sequences.
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.cam(x)
        x = self.eam(x)
        x = self.global_avg_pool(x)
        x = x.view(x.shape[0], -1)
        x = self.dropout(x)
        x = self.fc(x)
        x = self.fc_activation(x)
        return self.output_layer(x)  # RAW logits -- BCEWithLogitsLoss expects these, not sigmoid(logits)

    def predict_proba(self, x):
        """sigmoid(forward(x)) -- for inference/thresholding, never inside a training loop's backward pass."""
        with torch.no_grad():
            return torch.sigmoid(self.forward(x))


def build_ma1dcnn(config):
    """Construct an MA1DCNN from a plain hyperparameter dict (e.g. BEFORE_TUNING_CONFIG)."""
    return MA1DCNN(
        n_input_channels=config["n_input_channels"], channel_1=config["channel_1"],
        channel_2=config["channel_2"], channel_3=config["channel_3"],
        kernel_size=config["kernel_size"], fc_units=config["fc_units"],
        dropout=config["dropout"], n_outputs=config["n_outputs"],
        conv_dropout=config.get("conv_dropout", 0.0),      # .get() -- NOT config[...] -- for backward compatibility
        use_batchnorm=config.get("use_batchnorm", False),
    )


Writing /content/edge_fault_dx/models/ma1dcnn.py


In [6]:
%%writefile /content/edge_fault_dx/training/train.py
"""Training loop for MA1DCNN (Phase 5) -- edge-ai-fault-diagnosis-aerospace.

Ported from train.ipynb, where every function below was validated against its
own synthetic self-checks (see run_self_checks() / --self-check) and then run
for real against N-CMAPSS windowed data. The judgment calls behind each named
config -- why regularization, why AdamW was tried and rejected, why
conv-stack dropout, why BatchNorm1d is a real trade-off not a clean win, and
the 30-trial Optuna search that produced trial 13 -- are documented in
overfitting_results_and_options.docx, ma1dcnn_challenges_results_optuna.docx,
and overfitting_adamw_comparison.xlsx ("Optuna Search Results" / "Final
Verdict" sheets). This file carries no narrative, only the code and the
pointers to where the narrative lives.

Default config: TRIAL_13_MODEL_CONFIG / TRIAL_13_TRAIN_CONFIG -- the Optuna
search's Pareto-front knee-point pick (trial 13 of 30, see
optuna_search.py / optuna_search.ipynb). Per this project's "prove it
standalone first" discipline, trial 13 was not ported here until it had been
confirmed non-overfitting across two genuinely independent real re-runs of
this exact run_training(), in separate Colab sessions, plus the original
Optuna search's own (shorter) evaluation:

    source                 val_f1_macro   best epoch kept
    Optuna search           0.8375        n/a (search's own objective, not this run_training())
    Independent re-run A    0.8450        5
    Independent re-run B    0.8440        6   <- this checkpoint's weights (ma1dcnn_trial13.pt)

The two independent re-runs agree within 0.001 macro-F1 of each other -- a
real non-overfitting confirmation. (Correction: an earlier version of this
docstring cited 0.8401 and described "Re-run 1 (epoch 8): 0.8412" /
"Re-run 2 (epoch 5): 0.8450" as two separate re-runs. Checking the actual
executed notebook cells directly showed epochs 5 and 8 were two epochs from
the SAME single run, not two independent re-runs, and 0.8401 did not match
any real executed number. The corrected picture above is confirmed against
actual executed Colab cell output, not narrative claims -- see
phase4_architecture.docx for the full correction.) A real caveat carried
over from re-run B's real metrics, not fixed here: trial 13's
threshold=0.4215 trades precision for recall (val_precision_macro 0.7689 /
val_recall_macro 0.9383) -- a deliberate decision about that trade-off, not
a silent default, belongs before this config is deployed.

Usage
-----
    python train.py                                  # trains trial 13 (default), saves ma1dcnn_trial13.pt
    python train.py --config batchnorm                # trains a different named config
    python train.py --config trial13 --epochs 10 --windowed-dir /path/to/windowed
    python train.py --list-configs                    # show every named config and exit
    python train.py --self-check                      # run the ported synthetic sanity checks, then exit

Windowed data
-------------
Expects the *_train_windowed.npz / *_val_windowed.npz files windowing.ipynb
produces, one pair per file in NCMAPSS_FILES, under --windowed-dir (or the
EDGE_FAULT_DX_WINDOWED_DIR environment variable). The notebook's real runs
used a Google Drive path inside Colab; that path is not meaningful outside
Colab, so this module takes the directory as a normal CLI argument instead.
"""
from __future__ import annotations

import argparse
import logging
import os
import sys
import tempfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, hamming_loss, jaccard_score, precision_score, recall_score

# MA1DCNN now lives in its own module (src/edge_fault_dx/models/ma1dcnn.py), not inline
# here -- see that file's docstring for why. Path-based, not cwd-based, so this import
# resolves correctly whether train.py is run directly (`python train.py`, this repo's own
# documented usage) or imported from elsewhere (`from train import ...`, as
# threshold_sweep.py and false_alarm_review.py both do) regardless of the caller's cwd.
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))  # -> src/edge_fault_dx/
from models.ma1dcnn import MA1DCNN, build_ma1dcnn  # noqa: E402 -- re-exported: `from train import build_ma1dcnn` (threshold_sweep.py, false_alarm_review.py) keeps working unchanged.

LOGGER = logging.getLogger("edge_fault_dx.training.train")


# =============================================================================
# 1. Constants shared with ma1dcnn.ipynb / windowing.ipynb
# =============================================================================

# Column order fixes which label column a fault flag lands in, and its length
# (5) fixes the model's output width. Repeated verbatim from loaders.ipynb /
# ma1dcnn.ipynb / windowing.ipynb -- do not reorder independently of those.
FAULT_COLUMNS = ["fan_fail", "lpc_fail", "hpc_fail", "hpt_fail", "lpt_fail"]
N_FAULT_COLUMNS = len(FAULT_COLUMNS)  # 5

# Same 9 files, same fixed order, repeated from loaders.ipynb / windowing.ipynb.
NCMAPSS_FILES = [
    "N-CMAPSS_DS01-005.h5", "N-CMAPSS_DS02-006.h5", "N-CMAPSS_DS03-012.h5",
    "N-CMAPSS_DS04.h5", "N-CMAPSS_DS05.h5", "N-CMAPSS_DS06.h5",
    "N-CMAPSS_DS07.h5", "N-CMAPSS_DS08a-009.h5", "N-CMAPSS_DS08c-008.h5",
]

# windowing.ipynb's own output directory is this module's input. Saved files
# are named "<file>_<split>_windowed.npz", which _windowed_file_path() below
# reproduces exactly. The notebook hardcoded a Colab Drive path here; this
# module takes it from an env var (with a plain relative fallback) and lets
# --windowed-dir override it per run -- see build_dataloaders()/run_training().
DEFAULT_WINDOWED_DIR = os.environ.get("EDGE_FAULT_DX_WINDOWED_DIR", "data/windowed")

# Repeated from ma1dcnn.ipynb. Fixed at 28 because windowing.ipynb's own
# INCLUDE_FLIGHT_CONDITIONS=False default was used to produce the real
# data/windowed/ files -- see train_design_questions.docx for why the
# 28-vs-32 question stays genuinely open elsewhere; it is windowing.ipynb's
# call, not something this module decides.
DEFAULT_N_INPUT_CHANNELS = 28

# Table 4.4 (Akindoju, 2025) -- architecture hyperparameters, repeated verbatim from ma1dcnn.ipynb.
BEFORE_TUNING_CONFIG = {
    "n_input_channels": DEFAULT_N_INPUT_CHANNELS,
    "channel_1": 32, "channel_2": 64, "channel_3": 128,
    "kernel_size": 7, "fc_units": 64, "dropout": 0.5,
    "n_outputs": N_FAULT_COLUMNS,
}
AFTER_TUNING_CONFIG = {
    "n_input_channels": DEFAULT_N_INPUT_CHANNELS,
    "channel_1": 32, "channel_2": 128, "channel_3": 128,
    "kernel_size": 5, "fc_units": 16, "dropout": 0.39919,
    "n_outputs": N_FAULT_COLUMNS,
}


# =============================================================================
# 2. Model -- now defined in models/ma1dcnn.py (see this file's import above);
# MA1DCNN and build_ma1dcnn are re-exported from there, not redefined here.
# =============================================================================


# =============================================================================
# 3. Training-level constants -- repeated verbatim from train.ipynb Section 3
# =============================================================================

TRAIN_BEFORE_TUNING_CONFIG = {
    "learning_rate": 1e-3, "weight_decay": 1e-4, "batch_size": 128,
    "threshold": 0.5, "use_focal": False,
}
TRAIN_AFTER_TUNING_CONFIG = {
    "learning_rate": 0.00019963, "weight_decay": 4.29e-06, "batch_size": 256,
    "threshold": 0.76957, "use_focal": False,
}
# Same architecture/base settings as TRAIN_BEFORE_TUNING_CONFIG, weight_decay raised
# 10x -- added after the real baseline run showed val_loss rising from epoch 1 while
# train_loss kept falling (immediate overfitting), to test stronger L2 alone.
TRAIN_REGULARIZED_CONFIG = {
    "learning_rate": 1e-3, "weight_decay": 1e-3, "batch_size": 128,
    "threshold": 0.5, "use_focal": False,
}

# Same architecture as BEFORE_TUNING_CONFIG with dropout added inside the conv stack
# (0.2 on each of the 3 conv blocks), not just once before the FC layer. Neither
# coupled Adam nor decoupled AdamW weight_decay alone closed the generalization gap --
# see overfitting_results_and_options.docx.
BEFORE_TUNING_CONFIG_CONV_DROPOUT = dict(BEFORE_TUNING_CONFIG, conv_dropout=0.2)

# Same architecture as BEFORE_TUNING_CONFIG_CONV_DROPOUT, with BatchNorm1d added after
# each conv block (LAYERED ON TOP of conv_dropout=0.2, not replacing it).
BEFORE_TUNING_CONFIG_CONV_DROPOUT_BN = dict(BEFORE_TUNING_CONFIG_CONV_DROPOUT, use_batchnorm=True)

# CosineAnnealingWarmRestarts is confirmed by Akindoju's own abstract; T_0/T_mult are
# this project's own reasonable defaults, not stated anywhere in the praxis.
SCHEDULER_T_0 = 10
SCHEDULER_T_MULT = 2

N_EPOCHS = 30                 # upper bound; early stopping usually ends a run sooner
EARLY_STOPPING_PATIENCE = 5   # epochs allowed with no val_f1_macro improvement before stopping
RANDOM_SEED = 0                # for reproducible weight init and training order

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Trial 13 from optuna_search.py's completed 30-trial multi-objective search
# (directions=["maximize", "minimize"] over val_f1_macro / max generalization gap, TPE
# sampler, N_SEEDS_PER_TRIAL=2, real N-CMAPSS windowed data). study.best_trials /
# print_pareto_front(study) returned a 5-trial Pareto front (trials 9, 1, 12, 13, 4);
# trial 13 is the recommended pick -- a knee-point calculation (Branke, Deb, Dierolf, &
# Osswald, 2004) over the front's marginal F1-per-gap exchange rate shows a sharp drop
# right after it. Both trial 13 (val_f1_macro=0.8375, max_gap=-0.0188) and trial 12
# (0.8445, -0.0089) Pareto-dominate every hand-run config above outright -- see
# overfitting_adamw_comparison.xlsx's "Optuna Search Results" sheet for the full front
# and the exchange-rate math. channel_1=32 and channel_3=128 were held fixed by the
# search (not part of its space), inherited from BEFORE_TUNING_CONFIG unchanged, same
# as every Optuna trial. use_focal was never part of the search space either -- False,
# matching every config above.
TRIAL_13_MODEL_CONFIG = dict(
    BEFORE_TUNING_CONFIG,
    channel_2=64, kernel_size=5, fc_units=16,
    dropout=0.5945728900118123,
    conv_dropout=0.33869914326562556,
    use_batchnorm=False,
)
TRIAL_13_TRAIN_CONFIG = {
    "learning_rate": 0.0004689002238683733,
    "weight_decay": 0.002164379191774169,
    "batch_size": 256,
    "threshold": 0.4214610774407138,  # see module docstring -- trades precision for recall, by design
    "use_focal": False,
}

# Named-config registry, keyed by the --config CLI flag. "trial13" is the default --
# the only config confirmed non-overfitting across 3 independent evaluations (module
# docstring). The others stay here as a faithful record of the full sweep documented
# in overfitting_adamw_comparison.xlsx, selectable for comparison/reproduction, not as
# equally-recommended alternatives.
CONFIGS = {
    "baseline": {
        "model_config": BEFORE_TUNING_CONFIG,
        "train_config": TRAIN_BEFORE_TUNING_CONFIG,
        "optimizer_cls": torch.optim.Adam,
    },
    "regularized": {
        "model_config": BEFORE_TUNING_CONFIG,
        "train_config": TRAIN_REGULARIZED_CONFIG,
        "optimizer_cls": torch.optim.Adam,
    },
    "adamw": {
        "model_config": BEFORE_TUNING_CONFIG,
        "train_config": TRAIN_REGULARIZED_CONFIG,
        "optimizer_cls": torch.optim.AdamW,  # the regularizer that made the gap WORSE -- kept for the record, not recommended
    },
    "conv_dropout": {
        "model_config": BEFORE_TUNING_CONFIG_CONV_DROPOUT,
        "train_config": TRAIN_REGULARIZED_CONFIG,
        "optimizer_cls": torch.optim.Adam,
    },
    "batchnorm": {
        "model_config": BEFORE_TUNING_CONFIG_CONV_DROPOUT_BN,
        "train_config": TRAIN_REGULARIZED_CONFIG,
        "optimizer_cls": torch.optim.Adam,
    },
    "trial13": {
        "model_config": TRIAL_13_MODEL_CONFIG,
        "train_config": TRIAL_13_TRAIN_CONFIG,
        "optimizer_cls": torch.optim.Adam,
    },
}
DEFAULT_CONFIG_NAME = "trial13"


# =============================================================================
# 4. WindowedFaultDataset -- load and concatenate all 9 files' windows
# =============================================================================

def _windowed_file_path(filename, windowed_dir, split):
    """Reproduce windowing.ipynb's own save-path convention exactly: '<file>_<split>_windowed.npz'."""
    out_name = filename.replace(".h5", f"_{split}_windowed.npz")
    return os.path.join(windowed_dir, out_name)


class WindowedFaultDataset(torch.utils.data.Dataset):
    """All 9 files' windows for ONE split (train or val), concatenated into a single dataset.

    Loaded fully into memory at construction time -- windowing.ipynb's real output is
    roughly 710k train windows x 28 channels x 50 timesteps, a few GB as float32, well
    within a normal training host's RAM. If this ever stops being true, this is the
    class to make lazy/chunked.
    """

    def __init__(self, windowed_dir=DEFAULT_WINDOWED_DIR, split="train", files=None):
        files = files if files is not None else NCMAPSS_FILES
        X_parts, y_parts, unit_id_parts, start_cycle_parts = [], [], [], []

        for filename in files:
            path = _windowed_file_path(filename, windowed_dir, split)
            with np.load(path) as npz:
                X_parts.append(npz["X"])              # (n_windows_i, 28, 50)
                y_parts.append(npz["y"])              # (n_windows_i, 5)
                unit_id_parts.append(npz["unit_id"])  # (n_windows_i,) -- traceability only
                start_cycle_parts.append(npz["start_cycle"])

        # float32 for X (standard for model inputs) and for y too -- BCEWithLogitsLoss
        # needs a FLOAT target, not the int8 labels windowing.ipynb saved; casting once
        # here, at load time, means every batch downstream is already the right dtype.
        self.X = np.concatenate(X_parts, axis=0).astype(np.float32)
        self.y = np.concatenate(y_parts, axis=0).astype(np.float32)
        self.unit_id = np.concatenate(unit_id_parts, axis=0)
        self.start_cycle = np.concatenate(start_cycle_parts, axis=0)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        # torch.from_numpy shares memory with self.X/self.y rather than copying -- fine
        # here since DataLoader workers only ever read, never write, these tensors.
        return torch.from_numpy(self.X[idx]), torch.from_numpy(self.y[idx])


# =============================================================================
# 5. build_dataloaders -- the reusable loader-building function
# =============================================================================

def build_dataloaders(windowed_dir=DEFAULT_WINDOWED_DIR, batch_size=TRAIN_BEFORE_TUNING_CONFIG["batch_size"],
                       files=None, num_workers=0):
    """Build train and val DataLoaders over WindowedFaultDataset.

    num_workers=0 by default -- safe everywhere. Raise it if data loading becomes the
    training bottleneck on your hardware.
    """
    train_dataset = WindowedFaultDataset(windowed_dir=windowed_dir, split="train", files=files)
    val_dataset = WindowedFaultDataset(windowed_dir=windowed_dir, split="val", files=files)

    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)   # shuffled every epoch
    val_loader = torch.utils.data.DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)    # never shuffled
    return train_loader, val_loader


# =============================================================================
# 6. compute_metrics -- the five validation metrics
# =============================================================================

def compute_metrics(y_true, logits, threshold):
    """Threshold sigmoid(logits) at `threshold` and score against y_true.

    Parameters
    ----------
    y_true : torch.Tensor, shape (n, 5)
    logits : torch.Tensor, shape (n, 5) -- RAW logits, same convention as MA1DCNN.forward()
    threshold : float -- e.g. TRAIN_BEFORE_TUNING_CONFIG["threshold"] (0.5) or a tuned value
    """
    probs = torch.sigmoid(logits)                            # (n, 5), each entry independently in [0, 1]
    y_pred = (probs >= threshold).to(torch.int64).cpu().numpy()
    y_true_np = y_true.to(torch.int64).cpu().numpy()

    return {
        "f1_macro": f1_score(y_true_np, y_pred, average="macro", zero_division=0),
        "precision_macro": precision_score(y_true_np, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true_np, y_pred, average="macro", zero_division=0),
        "hamming_loss": hamming_loss(y_true_np, y_pred),      # fraction of (sample, label) entries that disagree
        "jaccard_macro": jaccard_score(y_true_np, y_pred, average="macro", zero_division=0),
    }


# =============================================================================
# 7. train_one_epoch -- one pass over the training set
# =============================================================================

def train_one_epoch(model, loader, optimizer, loss_fn, device):
    """One full pass over `loader`, updating `model`'s weights. Returns the average training loss."""
    model.train()  # enables dropout
    total_loss, n_batches = 0.0, 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()          # PyTorch accumulates gradients by default -- must clear them each step
        logits = model(X)              # (batch, 5) raw logits
        loss = loss_fn(logits, y)      # BCEWithLogitsLoss -- expects raw logits, applies sigmoid internally
        loss.backward()                # computes gradients for every parameter via autograd
        optimizer.step()                # applies one optimizer update using those gradients

        total_loss += loss.item()      # .item() pulls a plain Python float out of the 0-d loss tensor
        n_batches += 1

    return total_loss / n_batches


# =============================================================================
# 8. evaluate -- one pass over the validation set
# =============================================================================

def evaluate(model, loader, loss_fn, device, threshold):
    """One full pass over `loader` with gradients disabled. Returns a dict of metrics plus 'loss'."""
    model.eval()  # disables dropout -- deterministic forward passes for a fair evaluation
    total_loss, n_batches = 0.0, 0
    all_logits, all_y = [], []

    with torch.no_grad():  # no autograd bookkeeping needed -- saves memory and time during evaluation
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            loss = loss_fn(logits, y)
            total_loss += loss.item()
            n_batches += 1
            all_logits.append(logits.cpu())  # move off GPU immediately -- keeps GPU memory free during accumulation
            all_y.append(y.cpu())

    all_logits = torch.cat(all_logits, dim=0)  # every validation window's logits, in one tensor
    all_y = torch.cat(all_y, dim=0)
    metrics = compute_metrics(all_y, all_logits, threshold)  # computed ONCE, over the whole split
    metrics["loss"] = total_loss / n_batches
    return metrics


# =============================================================================
# 9. run_training -- the full loop, tying everything together
# =============================================================================

def run_training(model_config, train_config, windowed_dir=DEFAULT_WINDOWED_DIR, n_epochs=N_EPOCHS,
                  patience=EARLY_STOPPING_PATIENCE, device=DEVICE, files=None, verbose=True,
                  optimizer_cls=torch.optim.Adam):
    """Build everything, train with early stopping, return the BEST model and the full per-epoch history.

    Parameters
    ----------
    model_config : dict -- e.g. BEFORE_TUNING_CONFIG or TRIAL_13_MODEL_CONFIG, passed to build_ma1dcnn()
    train_config : dict -- e.g. TRAIN_BEFORE_TUNING_CONFIG or TRIAL_13_TRAIN_CONFIG
        (learning_rate, weight_decay, batch_size, threshold)
    optimizer_cls : type -- torch.optim.Adam (default) or torch.optim.AdamW. Adam applies
        weight_decay as coupled L2 (added into the gradient before Adam's adaptive
        per-parameter scaling); AdamW decouples it, applying weight_decay directly to the
        parameter instead (Loshchilov & Hutter, 2019). See overfitting_results_and_options.docx
        Section 4.2 and the "adamw" config above for why this was tried and not adopted.

    Returns
    -------
    model : MA1DCNN -- weights restored to the BEST epoch's checkpoint (highest val_f1_macro)
    history : list[dict] -- one record per epoch actually run: epoch, train_loss, and val_* for every metric
    """
    torch.manual_seed(RANDOM_SEED)  # reproducible weight init and (with shuffle=True) batch order

    train_loader, val_loader = build_dataloaders(
        windowed_dir=windowed_dir, batch_size=train_config["batch_size"], files=files)

    model = build_ma1dcnn(model_config).to(device)
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = optimizer_cls(model.parameters(), lr=train_config["learning_rate"],
                               weight_decay=train_config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=SCHEDULER_T_0, T_mult=SCHEDULER_T_MULT)

    history = []
    best_f1 = -1.0           # guaranteed to be beaten by epoch 1's real (>=0) f1_macro
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, n_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
        val_metrics = evaluate(model, val_loader, loss_fn, device, threshold=train_config["threshold"])
        scheduler.step()  # advances the cosine schedule -- called once per EPOCH, not per batch

        record = {"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in val_metrics.items()}}
        history.append(record)

        if verbose:
            LOGGER.info(
                "epoch %3d/%d | train_loss %.4f | val_loss %.4f | val_f1_macro %.4f | val_hamming %.4f",
                epoch, n_epochs, train_loss, val_metrics["loss"], val_metrics["f1_macro"], val_metrics["hamming_loss"],
            )

        if val_metrics["f1_macro"] > best_f1:  # strictly greater -- a TIE does not count as improvement
            best_f1 = val_metrics["f1_macro"]
            best_state = {k: v.clone() for k, v in model.state_dict().items()}  # clone -- a snapshot, not a live reference
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                if verbose:
                    LOGGER.info("early stopping: no val_f1_macro improvement for %d epochs", patience)
                break

    model.load_state_dict(best_state)  # return the BEST checkpoint, not whatever epoch the loop happened to end on
    return model, history


# =============================================================================
# 10. Synthetic self-checks -- ported from train.ipynb, run via --self-check
# =============================================================================

def _make_synthetic_windowed_files(windowed_dir, files, n_train, n_val, learnable=True, seed=0):
    rng = np.random.default_rng(seed)
    for filename in files:
        for split, n in [("train", n_train), ("val", n_val)]:
            if learnable:
                # Generative, not discriminative: draw y FIRST, then build channel j (j=0..4)
                # with a strong mean shift (+2.0 if y[:,j]==1 else -2.0) plus unit noise -- a
                # clear, reliably learnable signal for global-average-pooling + FC to pick up.
                y = rng.integers(0, 2, size=(n, 5)).astype(np.int8)
                X = rng.standard_normal((n, 28, 50)).astype(np.float32)
                for j in range(5):
                    shift = np.where(y[:, j] == 1, 2.0, -2.0).astype(np.float32)
                    X[:, j, :] += shift[:, None]
            else:
                # label has NOTHING to do with X -- unlearnable by construction.
                X = rng.standard_normal((n, 28, 50)).astype(np.float32)
                y = rng.integers(0, 2, size=(n, 5)).astype(np.int8)
            unit_id = np.arange(n)
            start_cycle = np.arange(n) * 50
            np.savez_compressed(_windowed_file_path(filename, windowed_dir, split),
                                 X=X, y=y, unit_id=unit_id, start_cycle=start_cycle)


def _check_windowed_fault_dataset_concatenates_all_files():
    with tempfile.TemporaryDirectory() as windowed_dir:
        files = ["FAKE_A.h5", "FAKE_B.h5"]
        sizes = {"FAKE_A.h5": 7, "FAKE_B.h5": 3}
        rng = np.random.default_rng(0)

        for filename, n in sizes.items():
            X = rng.random((n, 28, 50)).astype(np.float32)
            y = rng.integers(0, 2, size=(n, 5)).astype(np.int8)
            unit_id = np.arange(n)
            start_cycle = np.arange(n) * 50
            np.savez_compressed(_windowed_file_path(filename, windowed_dir, "train"),
                                 X=X, y=y, unit_id=unit_id, start_cycle=start_cycle)

        ds = WindowedFaultDataset(windowed_dir=windowed_dir, split="train", files=files)
        assert len(ds) == sum(sizes.values()), len(ds)
        assert ds.X.dtype == np.float32
        assert ds.y.dtype == np.float32

        x0, y0 = ds[0]
        assert x0.shape == (28, 50), x0.shape
        assert y0.shape == (5,), y0.shape
        assert x0.dtype == torch.float32
        assert y0.dtype == torch.float32

        x_last, _ = ds[len(ds) - 1]
        assert np.allclose(x_last.numpy(), ds.X[-1])

    LOGGER.info("WindowedFaultDataset: PASSED (concatenates all files, correct dtypes and shapes)")


def _check_build_dataloaders_shuffles_train_not_val():
    with tempfile.TemporaryDirectory() as windowed_dir:
        files = ["FAKE_A.h5"]
        rng = np.random.default_rng(1)
        n = 20
        for split in ["train", "val"]:
            X = rng.random((n, 28, 50)).astype(np.float32)
            y = rng.integers(0, 2, size=(n, 5)).astype(np.int8)
            np.savez_compressed(_windowed_file_path("FAKE_A.h5", windowed_dir, split),
                                 X=X, y=y, unit_id=np.arange(n), start_cycle=np.arange(n))

        train_loader, val_loader = build_dataloaders(windowed_dir=windowed_dir, batch_size=4, files=files)

        assert isinstance(train_loader.sampler, torch.utils.data.RandomSampler), type(train_loader.sampler)
        assert isinstance(val_loader.sampler, torch.utils.data.SequentialSampler), type(val_loader.sampler)
        assert len(train_loader.dataset) == n
        assert len(val_loader.dataset) == n

    LOGGER.info("build_dataloaders: PASSED (train uses RandomSampler, val uses SequentialSampler)")


def _check_compute_metrics_known_cases():
    y_true = torch.tensor([
        [1, 0, 1, 0, 1],
        [0, 1, 0, 1, 0],
        [1, 1, 0, 0, 1],
        [0, 0, 1, 1, 0],
    ], dtype=torch.float32)

    perfect_logits = torch.where(y_true == 1, torch.tensor(10.0), torch.tensor(-10.0))
    m = compute_metrics(y_true, perfect_logits, threshold=0.5)
    assert m["f1_macro"] == 1.0, m
    assert m["precision_macro"] == 1.0, m
    assert m["recall_macro"] == 1.0, m
    assert m["hamming_loss"] == 0.0, m
    assert m["jaccard_macro"] == 1.0, m

    wrong_logits = torch.where(y_true == 1, torch.tensor(-10.0), torch.tensor(10.0))
    m_wrong = compute_metrics(y_true, wrong_logits, threshold=0.5)
    assert m_wrong["f1_macro"] == 0.0, m_wrong
    assert m_wrong["hamming_loss"] == 1.0, m_wrong
    assert m_wrong["jaccard_macro"] == 0.0, m_wrong

    LOGGER.info("compute_metrics: PASSED (perfect prediction -> 1.0/0.0, fully inverted prediction -> 0.0/1.0)")


def _check_train_one_epoch_reduces_loss_on_a_tiny_batch():
    torch.manual_seed(0)
    config = dict(BEFORE_TUNING_CONFIG, channel_1=8, channel_2=16, channel_3=16, fc_units=8, dropout=0.0)
    model = build_ma1dcnn(config).to(DEVICE)

    X = torch.randn(16, 28, 50)
    y = (torch.rand(16, 5) > 0.5).float()
    loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X, y), batch_size=4)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
    loss_fn = nn.BCEWithLogitsLoss()

    first_loss = train_one_epoch(model, loader, optimizer, loss_fn, DEVICE)
    last_loss = first_loss
    for _ in range(19):
        last_loss = train_one_epoch(model, loader, optimizer, loss_fn, DEVICE)

    assert last_loss < first_loss, (first_loss, last_loss)
    assert last_loss == last_loss  # NaN != NaN -- this also fails loudly if training diverged

    LOGGER.info("train_one_epoch: PASSED (loss fell from %.4f to %.4f over 20 epochs on a fixed tiny batch)",
                first_loss, last_loss)


def _check_evaluate_returns_expected_keys_and_ranges():
    torch.manual_seed(0)
    model = build_ma1dcnn(dict(BEFORE_TUNING_CONFIG, channel_1=8, channel_2=16, channel_3=16, fc_units=8)).to(DEVICE)
    X = torch.randn(13, 28, 50)  # 13 -- deliberately not a multiple of batch_size, to exercise a ragged last batch
    y = (torch.rand(13, 5) > 0.5).float()
    loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X, y), batch_size=4)

    metrics = evaluate(model, loader, nn.BCEWithLogitsLoss(), DEVICE, threshold=0.5)

    expected_keys = {"f1_macro", "precision_macro", "recall_macro", "hamming_loss", "jaccard_macro", "loss"}
    assert set(metrics.keys()) == expected_keys, metrics.keys()
    for key in ["f1_macro", "precision_macro", "recall_macro", "hamming_loss", "jaccard_macro"]:
        assert 0.0 <= metrics[key] <= 1.0, (key, metrics[key])
    assert metrics["loss"] > 0, metrics["loss"]

    LOGGER.info("evaluate: PASSED (expected keys present, all metrics in [0,1], handles a ragged final batch)")


def _check_run_training_learns_a_separable_synthetic_task():
    with tempfile.TemporaryDirectory() as windowed_dir:
        files = ["FAKE_A.h5", "FAKE_B.h5"]
        _make_synthetic_windowed_files(windowed_dir, files, n_train=200, n_val=60, learnable=True, seed=0)

        small_model_config = dict(BEFORE_TUNING_CONFIG, channel_1=8, channel_2=16, channel_3=16, fc_units=8, dropout=0.1)
        fast_train_config = dict(TRAIN_BEFORE_TUNING_CONFIG, learning_rate=1e-2, weight_decay=0.0, batch_size=32)

        model, history = run_training(small_model_config, fast_train_config, windowed_dir=windowed_dir,
                                       n_epochs=15, patience=5, files=files, verbose=True)

        assert 1 <= len(history) <= 15
        best_f1_from_history = max(h["val_f1_macro"] for h in history)
        assert best_f1_from_history > 0.7, (
            f"expected run_training to learn the separable synthetic task well, "
            f"got best val_f1_macro={best_f1_from_history}")

        _, val_loader = build_dataloaders(windowed_dir=windowed_dir, batch_size=32, files=files)
        recheck = evaluate(model, val_loader, nn.BCEWithLogitsLoss(), DEVICE, threshold=fast_train_config["threshold"])
        assert abs(recheck["f1_macro"] - best_f1_from_history) < 1e-6, (recheck["f1_macro"], best_f1_from_history)

    LOGGER.info("run_training: PASSED (learned separable synthetic task, best val_f1_macro=%.4f, "
                "returned model confirmed to be the best checkpoint not just the last epoch)", best_f1_from_history)


def _check_run_training_early_stopping_triggers():
    with tempfile.TemporaryDirectory() as windowed_dir:
        files = ["FAKE_A.h5"]
        _make_synthetic_windowed_files(windowed_dir, files, n_train=40, n_val=20, learnable=True, seed=1)

        small_model_config = dict(BEFORE_TUNING_CONFIG, channel_1=8, channel_2=16, channel_3=16, fc_units=8)
        # learning_rate=0.0 -- weights never move, so val_f1_macro is IDENTICAL every epoch
        # after the first. With strict ">" for "improved", epoch 2 onward never counts as an
        # improvement, so patience=2 must stop training at exactly epoch 3.
        frozen_train_config = dict(TRAIN_BEFORE_TUNING_CONFIG, learning_rate=0.0, weight_decay=0.0, batch_size=8)

        model, history = run_training(small_model_config, frozen_train_config, windowed_dir=windowed_dir,
                                       n_epochs=10, patience=2, files=files, verbose=True)

        assert len(history) == 3, f"expected early stopping at exactly epoch 3, got {len(history)} epochs"

    LOGGER.info("run_training: PASSED (early stopping triggers at exactly the configured patience)")


def run_self_checks():
    """Run every synthetic sanity check ported from train.ipynb, in order. Raises on first failure."""
    _check_windowed_fault_dataset_concatenates_all_files()
    _check_build_dataloaders_shuffles_train_not_val()
    _check_compute_metrics_known_cases()
    _check_train_one_epoch_reduces_loss_on_a_tiny_batch()
    _check_evaluate_returns_expected_keys_and_ranges()
    _check_run_training_learns_a_separable_synthetic_task()
    _check_run_training_early_stopping_triggers()
    LOGGER.info("all self-checks PASSED")


# =============================================================================
# 11. CLI entry point
# =============================================================================

def _build_arg_parser():
    parser = argparse.ArgumentParser(
        description="Train MA1DCNN against real N-CMAPSS windowed data (default config: trial13).")
    parser.add_argument("--config", default=DEFAULT_CONFIG_NAME, choices=sorted(CONFIGS),
                         help=f"named config to train (default: {DEFAULT_CONFIG_NAME})")
    parser.add_argument("--windowed-dir", default=DEFAULT_WINDOWED_DIR,
                         help="directory containing windowing.ipynb's *_train_windowed.npz / "
                              "*_val_windowed.npz files (default: %(default)s, or $EDGE_FAULT_DX_WINDOWED_DIR)")
    parser.add_argument("--epochs", type=int, default=N_EPOCHS, help="max epochs (default: %(default)s)")
    parser.add_argument("--patience", type=int, default=EARLY_STOPPING_PATIENCE,
                         help="early-stopping patience in epochs (default: %(default)s)")
    parser.add_argument("--output", default=None,
                         help="checkpoint path to save to (default: ma1dcnn_<config>.pt)")
    parser.add_argument("--device", default=None, help="override auto-detected device, e.g. 'cpu' or 'cuda:0'")
    parser.add_argument("--list-configs", action="store_true", help="print every named config and exit")
    parser.add_argument("--self-check", action="store_true",
                         help="run the ported synthetic sanity checks against temp data, then exit")
    parser.add_argument("--quiet", action="store_true", help="suppress per-epoch logging")
    return parser


def main(argv=None):
    args = _build_arg_parser().parse_args(argv)
    logging.basicConfig(level=logging.WARNING if args.quiet else logging.INFO, format="%(message)s", stream=sys.stdout)

    if args.list_configs:
        for name, cfg in CONFIGS.items():
            marker = " (default)" if name == DEFAULT_CONFIG_NAME else ""
            LOGGER.info("%s%s: optimizer=%s, lr=%s, weight_decay=%s, batch_size=%s, threshold=%s",
                        name, marker, cfg["optimizer_cls"].__name__,
                        cfg["train_config"]["learning_rate"], cfg["train_config"]["weight_decay"],
                        cfg["train_config"]["batch_size"], cfg["train_config"]["threshold"])
        return

    if args.self_check:
        run_self_checks()
        return

    cfg = CONFIGS[args.config]
    device = torch.device(args.device) if args.device else DEVICE
    LOGGER.info("training config=%s on device=%s", args.config, device)

    model, history = run_training(
        model_config=cfg["model_config"], train_config=cfg["train_config"],
        windowed_dir=args.windowed_dir, n_epochs=args.epochs, patience=args.patience,
        device=device, verbose=not args.quiet, optimizer_cls=cfg["optimizer_cls"],
    )

    output_path = Path(args.output) if args.output else Path(f"ma1dcnn_{args.config}.pt")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), output_path)

    best = max(history, key=lambda h: h["val_f1_macro"])
    LOGGER.info(
        "saved %s model weights to %s (best epoch %d/%d: val_f1_macro=%.4f, val_loss=%.4f, "
        "gap=%.4f)", args.config, output_path, best["epoch"], len(history), best["val_f1_macro"],
        best["val_loss"], best["val_loss"] - history[best["epoch"] - 1]["train_loss"],
    )


if __name__ == "__main__":
    main()


Writing /content/edge_fault_dx/training/train.py


In [7]:
import sys
sys.path.insert(0, "/content/edge_fault_dx/training")
from train import TRIAL_13_TRAIN_CONFIG, build_dataloaders, compute_metrics, evaluate

print("imported evaluate / build_dataloaders / compute_metrics / TRIAL_13_TRAIN_CONFIG "
      "from train.py -- train.py itself was not modified.")


imported evaluate / build_dataloaders / compute_metrics / TRIAL_13_TRAIN_CONFIG from train.py -- train.py itself was not modified.


## 4. `resweep_threshold` — repeated verbatim from Section 10.3

Re-derives each reloaded checkpoint's best operating threshold fresh, rather than
trusting a number copied from last night's run -- cheap (forward passes only, no
training), and removes any dependency on last night's exact printed value still
being valid for this reloaded object.

In [8]:
def resweep_threshold(model, val_loader, device=None, thresholds=None):
    device = device or torch.device("cpu")
    if thresholds is None:
        thresholds = [round(0.05 + 0.025 * i, 4) for i in range(37)]   # 0.05 .. 0.95

    model.eval()
    all_logits, all_y = [], []
    with torch.no_grad():
        for X, y in val_loader:
            X = X.to(device)
            logits = model(X)
            all_logits.append(logits.cpu())
            all_y.append(y)
    all_logits = torch.cat(all_logits, dim=0)
    all_y = torch.cat(all_y, dim=0)

    best_t, best_f1 = None, -1.0
    rows = []
    for t in thresholds:
        m = compute_metrics(all_y, all_logits, threshold=t)
        rows.append((t, m["f1_macro"]))
        if m["f1_macro"] > best_f1:
            best_f1, best_t = m["f1_macro"], t

    return best_t, best_f1, rows


## 5. `count_params_any` + `benchmark_variant_quantized` — repeated verbatim from Section 11.1/11.2

`benchmark_variant()`'s own `.parameters()`-based param count silently undercounts a
quantized model's packed INT8 conv weights -- confirmed and fixed in the main
notebook's Section 11.1 (with its own self-checks there); not re-verified here,
just reused.

In [9]:
import io
import time
import numpy as np


def count_params_any(model):
    sd = model.state_dict()
    return sum(v.numel() for k, v in sd.items()
               if not (k.endswith(".scale") or k.endswith(".zero_point")))


def benchmark_variant_quantized(model, val_loader, threshold, n_warmup=10, n_timed=200, batch_size=1):
    cpu = torch.device("cpu")
    model_cpu = model.to(cpu)
    model_cpu.eval()
    metrics = evaluate(model_cpu, val_loader, nn.BCEWithLogitsLoss(), cpu, threshold=threshold)

    n_params = count_params_any(model_cpu)

    buf = io.BytesIO()
    torch.save(model_cpu.state_dict(), buf)
    size_mb = buf.tell() / (1024 ** 2)

    sample_x, _ = next(iter(val_loader))
    sample_x = sample_x[:batch_size].to(cpu)
    with torch.no_grad():
        for _ in range(n_warmup):
            model_cpu(sample_x)
        times_ms = np.empty(n_timed)
        for i in range(n_timed):
            t0 = time.perf_counter()
            model_cpu(sample_x)
            times_ms[i] = (time.perf_counter() - t0) * 1000.0

    return {
        "macro_f1": metrics["f1_macro"],
        "params": n_params,
        "size_mb": size_mb,
        "latency_mean_ms": float(times_ms.mean()),
        "latency_std_ms": float(times_ms.std()),
        "latency_p50_ms": float(np.percentile(times_ms, 50)),
        "latency_p95_ms": float(np.percentile(times_ms, 95)),
        "latency_p99_ms": float(np.percentile(times_ms, 99)),
        "n_timed_reps": n_timed,
        "num_threads": torch.get_num_threads(),
    }


## 6. Load the real QAT checkpoints from Drive and benchmark them (Step 6)

`torch.save(qat_model, path)` (what Section 10.7 actually does -- pickling the whole
already-converted module, not just its `state_dict`) turns out to reliably corrupt
one specific thing on reload: the quantized `Conv1d` submodules inside `conv1`/
`conv2`/`conv3` come back from `torch.load` missing ALL of `nn.Module`'s own
bookkeeping dict attributes (`_parameters`, `_buffers`, `_modules`,
`_backward_hooks`, and about a dozen others) -- not just the one that happened to
surface first as `AttributeError: 'Conv1d' object has no attribute
'_backward_hooks'`. This isn't an occasional, same-process-only glitch: confirmed by
saving in one Python process, fully exiting it, then loading in a completely
separate fresh process (the real overnight scenario) -- it reproduced every time.

The actual INT8 weights and quantization parameters are NOT affected -- only this
bookkeeping. So `repair_missing_module_dicts()` below just re-creates each missing
dict empty (exactly what a normally constructed module would have; nothing
meaningful was ever stored in them to lose), confirmed by direct testing to restore
a forward pass that is bit-identical to the model's own output from before it was
ever saved -- not merely "doesn't crash anymore," genuinely numerically correct.

A per-ratio `try/except` still wraps load + repair + a one-sample sanity probe, as
defense in depth -- if a checkpoint fails for some OTHER reason, that ratio is
skipped with a clear message instead of crashing the whole cell.

For reference, last night's run (Section 10.7, before this reload) reported
macro_f1 = 0.8385 / 0.8306 / 0.8202 for ratios 20% / 40% / 60% at re-swept
thresholds 0.425 / 0.425 / 0.400 -- printed below as a cross-check, not assumed.

In [10]:
from collections import OrderedDict

# The exact set of nn.Module bookkeeping dict attributes that can go missing from a
# quantized submodule after torch.save(whole_module) / torch.load() -- see this
# section's markdown above. Order does not matter; completeness does.
_REQUIRED_MODULE_DICT_ATTRS = [
    "_parameters", "_buffers", "_non_persistent_buffers_set", "_backward_hooks",
    "_is_full_backward_hook", "_forward_hooks", "_forward_hooks_with_kwargs",
    "_forward_hooks_always_called", "_forward_pre_hooks", "_forward_pre_hooks_with_kwargs",
    "_backward_pre_hooks", "_state_dict_hooks", "_state_dict_pre_hooks",
    "_load_state_dict_pre_hooks", "_load_state_dict_post_hooks", "_modules",
]


def repair_missing_module_dicts(model):
    """Walks every submodule and re-creates (empty) any of nn.Module's own
    bookkeeping dict attributes missing from its __dict__ after unpickling. See
    this section's markdown for why this is safe (nothing meaningful was ever
    stored in these particular dicts to lose) and confirmed correct (bit-identical
    forward pass vs. a never-pickled reference). Returns the list of attributes it
    had to patch, purely for visibility into whether/how often this fires.
    """
    repaired = []
    for name, m in model.named_modules():
        for attr in _REQUIRED_MODULE_DICT_ATTRS:
            if attr not in m.__dict__:
                object.__setattr__(m, attr, OrderedDict())
                repaired.append(f"{name or '<root>'}.{attr}")
    return repaired


QAT_CHECKPOINT_PATHS = {
    0.2: f"{DATA_DIR}/ma1dcnn_trial13_pruned20_quantized_qat.pt",
    0.4: f"{DATA_DIR}/ma1dcnn_trial13_pruned40_quantized_qat.pt",
    0.6: f"{DATA_DIR}/ma1dcnn_trial13_pruned60_quantized_qat.pt",
}
# Section 10.7's own reported macro-F1 from last night's successful run -- the
# cross-check reference below, not something this cell trusts blindly.
QAT_SECTION_10_7_F1 = {0.2: 0.8385, 0.4: 0.8306, 0.6: 0.8202}

_, qstep6_val_loader = build_dataloaders(
    windowed_dir=REAL_WINDOWED_DIR, batch_size=TRIAL_13_TRAIN_CONFIG["batch_size"])

quant_benchmark_results = {}   # ratio -> dict from benchmark_variant_quantized

for ratio, path in QAT_CHECKPOINT_PATHS.items():
    try:
        # weights_only=False: our own checkpoint, trusted source.
        qat_model = torch.load(path, map_location="cpu", weights_only=False)
        _repaired = repair_missing_module_dicts(qat_model)
        if _repaired:
            print(f"ratio={ratio:.0%} -- repaired {len(_repaired)} missing nn.Module "
                  f"bookkeeping attribute(s) after unpickling (see this section's "
                  f"markdown) -- does not affect the real weights.")
        with torch.no_grad():
            _probe_x, _ = next(iter(qstep6_val_loader))
            qat_model(_probe_x[:1].to("cpu"))
    except Exception as exc:
        print(f"ratio={ratio:.0%} -- QAT checkpoint did not load/run cleanly even "
              f"after the repair pass ({type(exc).__name__}: {exc}). Skipping this ratio.")
        continue

    best_t, best_f1, _ = resweep_threshold(qat_model, qstep6_val_loader)
    result = benchmark_variant_quantized(qat_model, qstep6_val_loader, best_t)
    quant_benchmark_results[ratio] = result

    print(f"ratio={ratio:.0%} + QAT -- re-swept threshold={best_t:.3f}, "
          f"macro_f1={result['macro_f1']:.4f} "
          f"(last night's Section 10.7 reported {QAT_SECTION_10_7_F1[ratio]:.4f}), "
          f"params={result['params']:,}, size={result['size_mb']:.3f} MB, "
          f"CPU latency p50/p95/p99={result['latency_p50_ms']:.3f} / "
          f"{result['latency_p95_ms']:.3f} / {result['latency_p99_ms']:.3f} ms "
          f"({result['num_threads']} threads)")

quant_benchmark_results  # ratio -> dict (macro_f1, params, size_mb, latency_*) -- fills in Section 6's "pending (Step 6)" cells


/usr/local/lib/python3.13/dist-packages/torch/_utils.py:476: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  device=storage.device,


ratio=20% -- repaired 48 missing nn.Module bookkeeping attribute(s) after unpickling (see this section's markdown) -- does not affect the real weights.
ratio=20% + QAT -- re-swept threshold=0.425, macro_f1=0.8385 (last night's Section 10.7 reported 0.8385), params=43,522, size=0.074 MB, CPU latency p50/p95/p99=0.485 / 0.611 / 0.793 ms (4 threads)
ratio=40% -- repaired 48 missing nn.Module bookkeeping attribute(s) after unpickling (see this section's markdown) -- does not affect the real weights.
ratio=40% + QAT -- re-swept threshold=0.425, macro_f1=0.8306 (last night's Section 10.7 reported 0.8306), params=25,467, size=0.049 MB, CPU latency p50/p95/p99=0.481 / 0.571 / 0.621 ms (4 threads)
ratio=60% -- repaired 48 missing nn.Module bookkeeping attribute(s) after unpickling (see this section's markdown) -- does not affect the real weights.
ratio=60% + QAT -- re-swept threshold=0.400, macro_f1=0.8202 (last night's Section 10.7 reported 0.8202), params=12,486, size=0.029 MB, CPU latency p5

{0.2: {'macro_f1': 0.8384678593060328,
  'params': 43522,
  'size_mb': 0.07428836822509766,
  'latency_mean_ms': 0.5040672500030041,
  'latency_std_ms': 0.09050727846480869,
  'latency_p50_ms': 0.48486950004189566,
  'latency_p95_ms': 0.6112914999278019,
  'latency_p99_ms': 0.7928904999835136,
  'n_timed_reps': 200,
  'num_threads': 4},
 0.4: {'macro_f1': 0.8305978526458055,
  'params': 25467,
  'size_mb': 0.048653602600097656,
  'latency_mean_ms': 0.4902133600069192,
  'latency_std_ms': 0.0402470453742167,
  'latency_p50_ms': 0.4808150000030764,
  'latency_p95_ms': 0.5707414999164938,
  'latency_p99_ms': 0.6206767000833222,
  'n_timed_reps': 200,
  'num_threads': 4},
 0.6: {'macro_f1': 0.8202488534655586,
  'params': 12486,
  'size_mb': 0.029366493225097656,
  'latency_mean_ms': 0.474858315004667,
  'latency_std_ms': 0.06477973099353337,
  'latency_p50_ms': 0.46095950000335506,
  'latency_p95_ms': 0.532415499981198,
  'latency_p99_ms': 0.8176368000931672,
  'n_timed_reps': 200,
  'num

In [11]:
import json

step6_out_path = f"{DATA_DIR}/step6_quant_benchmark_results.json"
with open(step6_out_path, "w") as f:
    json.dump({f"{ratio:.0%}": result for ratio, result in quant_benchmark_results.items()}, f, indent=2)

print(f"saved -> {step6_out_path}")

saved -> /content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/step6_quant_benchmark_results.json
